# TCP-Kalibrierung:
# Berechnung des Tool Center Points (TCP) mittels 4-Punkt-Methode

Diese Dokumentation beschreibt zwei Routinen zur Kalibrierung des TCPs eines Roboterwerkzeugs anhand von gemessenen Positionen und Orientierungen im Raum. Je nach verwendeter Funktion wird die Orientierung des Werkzeugs (tool0) unterschiedlich eingelesen.

<br>

### Eingabedaten

Die `daten`-Liste enthält Dictionaries mit folgenden Schlüsseln pro Messpunkt:

* **"Punkt":** Eine fortlaufende Nummer des Messpunkts.
* **"X", "Y", "Z":** Die kartesischen Koordinaten des TCPs in [mm], abgelesen im Roboter-Grundsystem (Wobj DrucktischV4).

**Orientierung via Quaternionen**

* **"Q1", "Q2", "Q3", "Q4":** Die Quaternionen im ABB-Format $[w, x, y, z]$. Die Norm des Quaternions sollte für eine valide Berechnung annähernd 1.0 sein.

<br>

### Mathematischer Ansatz (Least-Squares)

Die TCP-Berechnung basiert auf dem Prinzip, dass der TCP ein konstanter Punkt im Werkzeugkoordinatensystem ist. Wird das Werkzeug in verschiedene Positionen und Orientierungen gebracht, beschreibt der TCP immer den gleichen Punkt im Raum, wenn er im Werkzeugkoordinatensystem betrachtet wird.


Dieses wird mittels der Kleinste-Quadrate-Methode (`np.linalg.lstsq`) gelöst, um den optimalen $\text{TCP}_{\text{tool}}$-Vektor zu finden.

<br>

### Fehleranalyse und Qualitätsprüfung

Nach der Berechnung des TCPs wird für beide Varianten eine Fehleranalyse durchgeführt:

1. **Erwartete Position:** Berechnung der erwarteten TCP-Position im globalen Raum für jeden Messpunkt (`tip_positions`).
2. **Mittlerer Fixpunkt:** Bestimmung des mittleren Fixpunkts (`mean_tip`) über alle erwarteten TCP-Positionen.
3. **Einzelfehler:** Berechnung des Einzelfehlers (`errors`) als Abstand jedes erwarteten TCPs zum `mean_tip`.
4. **Validierung der Orientierung:** Überprüfung, ob die Norm annähernd 1.0 entspricht.


5. **Ausreißer:** Identifikation von Ausreißern basierend auf dem Einzelfehler im Verhältnis zum mittleren Fehler.
6. **Toleranzprüfung:** Prüfung, ob die berechneten Fixpunkte innerhalb eines definierten Toleranzbereichs um einen `TARGET_POS` (Erwarteter Fixpunkt) liegen und ob die X-, Y-, Z-Werte positiv sind.

<br>

### Ausgabe

Die Routinen geben folgende Informationen aus:

* Eine detaillierte Qualitätsprüfung für jeden einzelnen Messpunkt, inklusive möglicher Fehlermeldungen oder Warnungen.
* Den mittleren Gesamtfehler (Präzision der Messung).
* Den berechneten Fixpunkt im Raum (Wobj DrucktischV4).
* Den endgültigen, berechneten TCP-Vektor (X, Y, Z) im Werkzeugkoordinatensystem.
* Eine Formatierung des TCP für die direkte Übernahme in ABB RAPID Code.

<br>

---

### Verwendung

1. Wählen Sie die passende Funktion basierend auf Ihren Daten (Euler-Winkel oder Quaternionen).
2. Tragen Sie die gemessenen X, Y, Z Positionen sowie die Orientierungswerte (`Q1, Q2, Q3, Q4`) in die `daten`-Liste ein.
3. Führen Sie die Zelle aus, um den TCP zu berechnen und die Qualitätsprüfung zu erhalten.
4. Überprüfen Sie die "Qualitätsprüfung der Messpunkte" auf Warnungen oder Fehler.
5. Aktualisieren Sie in `CalibData.modx` den ausgegebenen "BERECHNETER TCP"-Wert für die Definition des TCPs in ABB RobotStudio.

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R

# ===================================================================
# DOKUMENTATION DER KOORDINATENSYSTEME (ABB-STANDARD)
# ===================================================================
# Bezugssystem (Wobj):  DrucktischV4  -> Alle X, Y, Z Werte beziehen sich auf diesen Nullpunkt
# Werkzeug (Tool):      tool0         -> Die Roboter-Flanschmitte (Kalibrier-Ausgangspunkt)
# Rotation:             Quaternionen  -> ABB-Format: [q1, q2, q3, q4] (w, x, y, z)
# ===================================================================


# Messdaten (Werte direkt vom FlexPendant abgelesen
daten = [
    {"Punkt": 1, "X": 395.13, "Y": 371.45, "Z": 207.86, "Q1": 0.23238, "Q2": -0.89551, "Q3": -0.13690, "Q4": 0.35401},
    {"Punkt": 2, "X": 196.78, "Y": 327.38, "Z": 185.37, "Q1": 0.17146, "Q2": 0.55312, "Q3": 0.80906, "Q4": -0.10040},
    {"Punkt": 3, "X": 454.46, "Y": 294.73, "Z": 153.70, "Q1": 0.23681, "Q2": 0.58165, "Q3": -0.77615, "Q4": -0.05649},
    {"Punkt": 4, "X": 351.88, "Y": 334.27, "Z": 236.03, "Q1": 0.01312, "Q2": 0.41379, "Q3": -0.84172, "Q4": 0.34658},
    #{"Punkt": 5, "X": , "Y": , "Z": , "Q1": , "Q2": , "Q3": , "Q4": },
    #{"Punkt": 8, "X": , "Y": , "Z": , "Q1": , "Q2": , "Q3": , "Q4": },
]

A = []
B = []

def abb_to_scipy_quat(d):
    # SciPy erwartet [x, y, z, w]. ABB liefert [w, x, y, z] als [Q1, Q2, Q3, Q4].
    return [d["Q2"], d["Q3"], d["Q4"], d["Q1"]]

# Referenzpunkt 0 vorbereiten
P0 = np.array([daten[0]["X"], daten[0]["Y"], daten[0]["Z"]])
R0 = R.from_quat(abb_to_scipy_quat(daten[0])).as_matrix()

# Gleichungssystem aufstellen
for i in range(1, len(daten)):
    Pi = np.array([daten[i]["X"], daten[i]["Y"], daten[i]["Z"]])
    Ri = R.from_quat(abb_to_scipy_quat(daten[i])).as_matrix()
    A.append(R0 - Ri)
    B.append(Pi - P0)

A = np.vstack(A)
B = np.concatenate(B)

# System lösen (Least-Squares)
tcp, _, _, _ = np.linalg.lstsq(A, B, rcond=None)

# --- FEHLERBERECHNUNG PRO PUNKT ---
tip_positions = []
for d in daten:
    Pi = np.array([d["X"], d["Y"], d["Z"]])
    Ri = R.from_quat(abb_to_scipy_quat(d)).as_matrix()
    tip = Pi + Ri.dot(tcp)
    tip_positions.append(tip)

mean_tip = np.mean(tip_positions, axis=0)
errors = np.linalg.norm(tip_positions - mean_tip, axis=1)
mean_error = np.mean(errors)

print("=== Qualitätsprüfung der Messpunkte ===")

target_x, target_y, target_z = 315.0, 315.0, 50.0
tolerance = 250.0

for i, err in enumerate(errors):
    status_list = []

    # 0. Prüfung der Quaternionen-Gültigkeit (Norm muss annähernd 1 sein)
    q_norm = np.linalg.norm([daten[i]["Q1"], daten[i]["Q2"], daten[i]["Q3"], daten[i]["Q4"]])
    if not np.isclose(q_norm, 1.0, atol=1e-3):
        status_list.append(f"❌ QUATERNION UNGÜLTIG (Norm={q_norm:.3f} statt 1.0)")

    # 1. Mathematische Qualitätsprüfung (Ausreißer-Erkennung via Least-Squares)
    if err > 1.5 and err > (mean_error * 2):
        status_list.append("⚠️ MÖGLICHER TIPPFEHLER IN DEN WERTEN")

    # Die berechnete Fixpunkt-Position für diesen spezifischen Punkt im Raum
    pt = tip_positions[i]

    # 2. Translatorische Prüfung: X-Achse
    if pt[0] < 0:
        status_list.append("❌ X IST NEGATIV")
    elif abs(pt[0] - target_x) > tolerance:
        status_list.append(f"⚠️ X AUßERHALB BEREICH ({pt[0]:.1f}mm)")

    # 3. Translatorische Prüfung: Y-Achse
    if pt[1] < 0:
        status_list.append("❌ Y IST NEGATIV")
    elif abs(pt[1] - target_y) > tolerance:
        status_list.append(f"⚠️ Y AUßERHALB BEREICH ({pt[1]:.1f}mm)")

    # 4. Translatorische Prüfung: Z-Achse
    if abs(pt[2] - target_z) > tolerance:
        status_list.append(f"⚠️ Z AUßERHALB BEREICH ({pt[2]:.1f}mm)")

    # Status-Ausgabe zusammenfassen
    if not status_list:
        status_str = "OK"
    else:
        status_str = " | ".join(status_list)

    print(f"Punkt {daten[i]['Punkt']}: Einzelfehler = {err:.3f} mm -> {status_str}")

print("\n=== Endergebnis ===")
print(f"Mittlerer Gesamtfehler (Präzision der Messung): {mean_error:.3f} mm")
print(f"Berechneter Fixpunkt im Raum (Wobj DrucktischV4): X={mean_tip[0]:.2f}, Y={mean_tip[1]:.2f}, Z={mean_tip[2]:.2f}\n")

print(f"*** BERECHNETER TCP: X={tcp[0]:.3f}, Y={tcp[1]:.3f}, Z={tcp[2]:.3f} ***\n")

# Ausgabe des TCPs im ABB RAPID Format
print("=== RAPID Ausgabe ===")
print(f"PERS tooldata Tool:=[TRUE,[[{tcp[0]:.3f},{tcp[1]:.3f},{tcp[2]:.3f}],[1,0,0,0]],...")

=== Qualitätsprüfung der Messpunkte ===
Punkt 1: Einzelfehler = 0.254 mm -> OK
Punkt 2: Einzelfehler = 0.611 mm -> OK
Punkt 3: Einzelfehler = 0.605 mm -> OK
Punkt 4: Einzelfehler = 0.311 mm -> OK

=== Endergebnis ===
Mittlerer Gesamtfehler (Präzision der Messung): 0.445 mm
Berechneter Fixpunkt im Raum (Wobj DrucktischV4): X=315.76, Y=315.72, Z=86.10

*** BERECHNETER TCP: X=-10.068, Y=103.644, Z=115.510 ***

=== RAPID Ausgabe ===
PERS tooldata Tool:=[TRUE,[[-10.068,103.644,115.510],[1,0,0,0]],...
